# Exercice 2 - Q-Learning avec FrozenLake-v1

Objectif : implémenter un premier algorithme de Q-Learning de zéro avec Gymnasium et NumPy.

## Étape 1 - Créer l'environnement et initialiser la Q-table

`FrozenLake-v1` possède un nombre fini d'états et d'actions. On peut donc représenter ce que l'agent apprend dans une Q-table.

In [ ]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", is_slippery=True)

number_of_states = env.observation_space.n
number_of_actions = env.action_space.n

q_table = np.zeros((number_of_states, number_of_actions))

print("Nombre d'états :", number_of_states)
print("Nombre d'actions :", number_of_actions)
print("Dimensions de la Q-table :", q_table.shape)
print(q_table)

## Étape 2 - Entraîner l'agent avec le Q-Learning

L'agent utilise une stratégie epsilon-greedy : il explore souvent au début, puis exploite progressivement les valeurs apprises dans la Q-table.

In [ ]:
training_episodes = 20_000
max_steps_per_episode = 100

learning_rate = 0.8
discount_factor = 0.95

epsilon = 1.0
min_epsilon = 0.01
epsilon_decay = 0.001

rng = np.random.default_rng(seed=42)

for episode in range(training_episodes):
    state, info = env.reset()
    terminated = False
    truncated = False

    for step in range(max_steps_per_episode):
        if rng.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(q_table[state, :]))

        new_state, reward, terminated, truncated, info = env.step(action)

        old_value = q_table[state, action]
        max_future_q = np.max(q_table[new_state, :])
        new_value = old_value + learning_rate * (
            reward + discount_factor * max_future_q - old_value
        )
        q_table[state, action] = new_value

        state = new_state

        if terminated or truncated:
            break

    epsilon = min_epsilon + (1.0 - min_epsilon) * np.exp(-epsilon_decay * episode)

print("Q-table entraînée :")
print(q_table)

## Étape 3 - Évaluer l'agent entraîné

Pendant l'évaluation, l'agent n'explore plus. Il choisit toujours l'action avec la meilleure valeur dans la Q-table.

In [ ]:
evaluation_episodes = 100
total_wins = 0

for episode in range(evaluation_episodes):
    state, info = env.reset()
    terminated = False
    truncated = False

    for step in range(max_steps_per_episode):
        action = int(np.argmax(q_table[state, :]))
        state, reward, terminated, truncated, info = env.step(action)

        if terminated or truncated:
            if reward == 1.0:
                total_wins += 1
            break

success_rate = total_wins / evaluation_episodes * 100
print(f"Taux de réussite sur {evaluation_episodes} épisodes : {success_rate:.0f}%")

env.close()